# AI Zero to Hero: End of Course Practical Assessment
## Practical Data Analysis: EDA, Predictive Modelling, and Hierarchical Clustering

Welcome to the end-of-course practical assessment! In this notebook, you will apply the concepts learned across the course using Python in Google Colab.

### Objectives:
1. **Mount Google Drive** and prepare your environment.
2. Perform **Exploratory Data Analysis (EDA)** on a real-world scientific dataset.
3. Build and evaluate **Predictive Regression and Classification Models**.
4. Apply **Hierarchical Agglomerative Clustering** and interpret dendrograms.

---

## Task 0: Setup and Mount Google Drive
Run the cell below to load the required Python libraries and mount your Google Drive.

In [ ]:
from google.colab import drive
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

# Mount Google Drive
drive.mount('/content/drive')

# Set up assessment folder
assessment_folder = '/content/drive/MyDrive/AI_Zero_To_Hero/Assessment'
os.makedirs(assessment_folder, exist_ok=True)
print("Setup complete. Saving folder initialized at:", assessment_folder)

--- 
# Section 1: Exploratory Data Analysis (EDA)

We will be working with the **Wine Dataset**, which contains chemical properties of wines derived from three different cultivars.
Below, we load the dataset into a Pandas DataFrame.

In [ ]:
# Load dataset
wine_raw = load_wine(as_frame=True)
df = wine_raw.frame.copy()

# Display first few rows
df.head()

### Code Example 1.1: Calculating Summary Statistics
Here is how we calculate the mean, standard deviation, and summary stats for the `alcohol` feature:

In [ ]:
mean_alc = df['alcohol'].mean()
std_alc = df['alcohol'].std()
print(f"Alcohol - Mean: {mean_alc:.2f}, SD: {std_alc:.2f}")
df[['alcohol']].describe()

### ✏️ Exercise 1.1
Modify the code below to calculate the mean, standard deviation, and summary statistics for the **`flavanoids`** variable instead of `alcohol`.

In [ ]:
# --- YOUR CODE HERE ---
# Change 'alcohol' to 'flavanoids'
mean_flav = df['alcohol'].mean()
std_flav = df['alcohol'].std()
print(f"Flavanoids - Mean: {mean_flav:.2f}, SD: {std_flav:.2f}")
df[['alcohol']].describe()

### Code Example 1.2: Scatter Plots and Saving Figures
We can explore relationships between numeric variables using scatter plots and save them to Google Drive.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df['flavanoids'], df['total_phenols'], color='purple', alpha=0.7)
plt.xlabel('Flavanoids')
plt.ylabel('Total Phenols')
plt.title('Flavanoids vs Total Phenols')
plt.grid(True, linestyle='--', alpha=0.5)

# Save plot
fig_path = os.path.join(assessment_folder, 'flavanoids_vs_phenols.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print('Figure saved to:', fig_path)

### ✏️ Exercise 1.2
Alter the code below to plot **`alcohol`** on the X-axis and **`color_intensity`** on the Y-axis. Change the point color to `'darkgreen'` and save the figure as `'alcohol_vs_color.png'`.

In [ ]:
# --- YOUR CODE HERE ---
plt.figure(figsize=(7, 5))
plt.scatter(df['flavanoids'], df['total_phenols'], color='purple', alpha=0.7) # Modify X, Y, and color
plt.xlabel('Alcohol')
plt.ylabel('Color Intensity')
plt.title('Alcohol vs Color Intensity')
plt.grid(True, linestyle='--', alpha=0.5)

fig_path = os.path.join(assessment_folder, 'flavanoids_vs_phenols.png') # Update filename
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

--- 
# Section 2: Predictive Modelling

## 2.1 Regression
Suppose we want to predict the continuous response variable **`total_phenols`** based on other chemical properties.

In [ ]:
# Target continuous variable
y_reg = df['total_phenols']

# Feature: Flavanoids
X_reg1 = df[['flavanoids']]

# Split into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(X_reg1, y_reg, test_size=0.25, random_state=42)

# Fit Simple Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)

print(f"Simple Regression R2 Score: {r2_score(y_test, y_pred):.3f}")
print(f"Simple Regression RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")

### ✏️ Exercise 2.1
Now perform a **Multiple Linear Regression** predicting `total_phenols` using **three features**: `['flavanoids', 'alcalinity_of_ash', 'ash']`.
Calculate and report the $R^2$ and RMSE on the test set. Did adding features improve performance?

In [ ]:
# --- YOUR CODE HERE ---
X_reg2 = df[['flavanoids']] # Alter this list to include all 3 features

X_train, X_test, y_train, y_test = train_test_split(X_reg2, y_reg, test_size=0.25, random_state=42)

mlr_model = LinearRegression()
mlr_model.fit(X_train, y_train)
y_pred_multi = mlr_model.predict(X_test)

print(f"Multiple Regression R2 Score: {r2_score(y_test, y_pred_multi):.3f}")
print(f"Multiple Regression RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_multi)):.3f}")

## 2.2 Classification
Now we want to predict the categorical cultivar class (`target`, values 0, 1, or 2) using Logistic Regression.

In [ ]:
X_clf = df.drop(columns=['target'])
y_clf = df['target']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.3, random_state=42)

# Standardize features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_c)
X_test_scaled = scaler.transform(X_test_c)

# Train Logistic Regression model
clf = LogisticRegression()
clf.fit(X_train_scaled, y_train_c)

y_clf_pred = clf.predict(X_test_scaled)
acc = accuracy_score(y_test_c, y_clf_pred)
print(f"Classification Accuracy: {acc * 100:.2f}%")

# Display Confusion Matrix
cm = confusion_matrix(y_test_c, y_clf_pred)
ConfusionMatrixDisplay(cm).plot()
plt.title('Wine Cultivar Classification Confusion Matrix')
plt.show()

### 💬 Reflection Question 2.1
Look at the confusion matrix above. Did the classifier make any misclassifications on the test set? If so, which classes were confused?

--- 
# Section 3: Hierarchical Agglomerative Clustering

In unsupervised learning, we group observations without using predefined ground-truth labels. Hierarchical Agglomerative Clustering builds a nested tree structure (dendrogram) based on pairwise distances.

### Code Example 3.1: Standardisation and Dendrogram Plotting
Distance metrics are sensitive to variable scales. Therefore, we standardize the numeric features before clustering.

In [ ]:
# Select numerical features (excluding target)
features = df.drop(columns=['target'])

# Scale dataset
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Perform Hierarchical Clustering using Ward linkage
linked_ward = linkage(scaled_features, method='ward')

# Plot Dendrogram
plt.figure(figsize=(12, 6))
dendrogram(linked_ward, truncate_mode='lastp', p=30, leaf_rotation=90., leaf_font_size=10.)
plt.title('Hierarchical Clustering Dendrogram (Ward Linkage)')
plt.xlabel('Cluster Sample Index / Subtree Size')
plt.ylabel('Distance (Dissimilarity)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### ✏️ Exercise 3.1
Change the linkage method in the code below from `'ward'` to **`'complete'`**. Run the cell and observe how the structure of the dendrogram changes.

In [ ]:
# --- YOUR CODE HERE ---
# Change method='ward' to method='complete'
linked_complete = linkage(scaled_features, method='ward')

plt.figure(figsize=(12, 6))
dendrogram(linked_complete, truncate_mode='lastp', p=30, leaf_rotation=90., leaf_font_size=10.)
plt.title('Hierarchical Clustering Dendrogram (Complete Linkage)')
plt.xlabel('Cluster Sample Index / Subtree Size')
plt.ylabel('Distance (Dissimilarity)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### Code Example 3.2: Cutting the Dendrogram into Clusters
We can cut the dendrogram to form $k=3$ distinct clusters and attach the assigned cluster IDs back to our dataset.

In [ ]:
# Extract 3 clusters from the Ward linkage tree
cluster_labels = fcluster(linked_ward, k=3, criterion='maxclust')

# Assign cluster IDs to the dataframe
df['Cluster_ID'] = cluster_labels

# Compare cluster IDs with ground-truth target cultivars
pd.crosstab(df['target'], df['Cluster_ID'], rownames=['Actual Cultivar'], colnames=['Assigned Cluster'])

### 💬 Reflection Question 3.1
Based on the cross-tabulation table above, how well did the unsupervised hierarchical clustering group the wines compared to their actual cultivar targets?

--- 
# Summary & Final Checklist

Congratulations on completing the assessment! Check that you have:
- [ ] Completed all code modifications in **Exercises 1.1, 1.2, 2.1, and 3.1**.
- [ ] Saved output plots and verified that they appear in your Google Drive assessment folder.
- [ ] Answered all text reflection questions in markdown.